<a href="https://colab.research.google.com/github/AkshatkMalik/python-data-science-toolkit/blob/main/Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ASSIGNMENT-11:Recommendation System**

**Task 1: Data Preprocessing**

In this task, we will load the Anime dataset with the help of pandas, examine the data structure, analyze the data and address the issue of missing data, for example, missing genres or ratings. We will inspect the data to understand how anime titles, types, genres, episodes, and ratings are organized.

In [2]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore', category=DeprecationWarning)

# Load the dataset into a pandas DataFrame
anime_df = pd.read_csv('anime.csv')

print(' Basic Data Exploration ')
print(f'Dataset Shape (Rows, Columns): {anime_df.shape}')
print('\nMissing Values per Column:\n', anime_df.isnull().sum())

print('\nFirst 5 Rows of Dataset:')
display(anime_df.head())

# Handle missing values: Drop rows with missing genres or ratings to maintain data quality
anime_clean = anime_df.dropna(subset=['genre', 'rating']).reset_index(
    drop=True
)
print(f'\nCleaned Dataset Shape: {anime_clean.shape}')

# Statistical Summary of Numerical Features
print('\nStatistical Summary:')
display(anime_clean[['rating', 'members']].describe())

 Basic Data Exploration 
Dataset Shape (Rows, Columns): (12294, 7)

Missing Values per Column:
 anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

First 5 Rows of Dataset:


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266



Cleaned Dataset Shape: (12017, 7)

Statistical Summary:


,rating,members
count,12017.000000,1.201700e+04
mean,6.478264,1.834888e+04
std,1.023857,5.537250e+04
min,1.670000,1.200000e+01
25%,5.890000,2.250000e+02
50%,6.570000,1.552000e+03
75%,7.180000,9.588000e+03
max,10.000000,1.013917e+06


**Task 2: Feature Extraction**

In this task, we need to choose the key features (genre and rating) on which we would base the similarity. Since genres are text strings, we can perform TF-IDF Vectorization and convert them into a numerical vector representation. Meanwhile, numerical features such as ratings or members need to be normalized if required.

In [3]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

#  Select 'genre' as our primary feature for content-based similarity
# TF-IDF (Term Frequency-Inverse Document Frequency) converts text genres into numerical vectors
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(anime_clean['genre'])

print('TF-IDF Matrix Shape:', tfidf_matrix.shape)
print(
    'Example feature representation for the first anime:',
    tfidf_matrix[0].toarray(),
)

TF-IDF Matrix Shape: (12017, 46)
Example feature representation for the first anime: [[0.         0.         0.         0.         0.         0.
  0.         0.         0.43900839 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.48892081 0.         0.52027478 0.         0.         0.
  0.         0.         0.         0.         0.         0.54547431
  0.         0.         0.         0.        ]]


**Task 3: Recommendation System (Cosine Similarity)**

In this task, we have to design a recommendation function based on the Cosine Similarity. For a given anime title we have to calculate its similarities with all the other animes and pick the most similar to it. We can also set a threshold value to pick only the most similar animes according to the similarity score.

In [8]:

from sklearn.metrics.pairwise import cosine_similarity

# Compute Cosine Similarity matrix across all anime
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('Cosine Similarity Matrix Shape:', cosine_sim.shape)


# Design recommendation function with min_similarity threshold parameter
def recommend_anime(
    title, min_similarity=0.0, top_n=5, cosine_sim=cosine_sim, df=anime_clean
):
  if title not in df['name'].values:
    return f"Anime '{title}' not found in dataset."

  idx = df[df['name'] == title].index[0]
  sim_scores = list(enumerate(cosine_sim[idx]))

  # Filter by minimum similarity threshold and sort in descending order
  filtered_scores = [
      (i, score) for i, score in sim_scores if score >= min_similarity
  ]
  filtered_scores = sorted(filtered_scores, key=lambda x: x[1], reverse=True)

  # Exclude the target anime itself if present at the top
  filtered_scores = [item for item in filtered_scores if item[0] != idx][:top_n]

  anime_indices = [i[0] for i in filtered_scores]
  similarity_values = [i[1] for i in filtered_scores]

  result_df = df[['name', 'genre', 'type', 'rating']].iloc[
      anime_indices
  ].copy()
  result_df['similarity_score'] = similarity_values
  return result_df


# Test the recommender
print(" Recommendations for 'Naruto' (Min Similarity >= 0.2) ")
display(recommend_anime('Naruto', min_similarity=0.2, top_n=5))

Cosine Similarity Matrix Shape: (12017, 12017)
--- Recommendations for 'Naruto' (Min Similarity >= 0.2) ---


,name,genre,type,rating,similarity_score
486,Boruto: Naruto the Movie,"Action, Comedy, Martial Arts, Shounen, Super P...",Movie,8.03,1.0
615,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,7.94,1.0
1103,Boruto: Naruto the Movie - Naruto ga Hokage ni...,"Action, Comedy, Martial Arts, Shounen, Super P...",Special,7.68,1.0
1343,Naruto x UT,"Action, Comedy, Martial Arts, Shounen, Super P...",OVA,7.58,1.0
1472,Naruto: Shippuuden Movie 4 - The Lost Tower,"Action, Comedy, Martial Arts, Shounen, Super P...",Movie,7.53,1.0


**Task 4: Evaluation and Performance Analysis**

In this task we consider the evaluation of content-based recommendation systems. While user-item train-test splits and evaluation metrics such as Precision, Recall, and F1-score are typical for many supervised learning tasks, unsupervised recommenders are traditionally assessed using user-relevance ranking measures (e.g., Precision@K, Recall@K), or a qualitative review of genre alignment.

In [9]:

from sklearn.model_selection import train_test_split

# Split dataset into training (80%) and testing (20%) sets
train_data, test_data = train_test_split(
    anime_clean, test_size=0.2, random_state=42
)

print(f'Training Set Size: {train_data.shape[0]} rows')
print(f'Testing Set Size: {test_data.shape[0]} rows')


# Evaluation helper function (Evaluating Precision & Recall based on genre overlap)
def evaluate_recommendation_quality(test_anime_title, recommended_df):
  # Get target anime genres
  target_genres = set(
      anime_clean[anime_clean['name'] == test_anime_title]['genre']
      .values[0]
      .split(', ')
  )

  matches = 0
  total_recommended = len(recommended_df)

  for _, row in recommended_df.iterrows():
    rec_genres = set(row['genre'].split(', '))
    # If there is genre overlap, count as relevant recommendation (true positive)
    if len(target_genres.intersection(rec_genres)) > 0:
      matches += 1

  precision = matches / total_recommended if total_recommended > 0 else 0
  # Recall estimated against target genre breadth
  recall = (
      matches / len(target_genres) if len(target_genres) > 0 else 0
  )
  f1 = (
      (2 * precision * recall) / (precision + recall)
      if (precision + recall) > 0
      else 0
  )

  print(f'Evaluation Metrics for recommendations of "{test_anime_title}":')
  print(f'  - Precision: {precision:.2f}')
  print(f'  - Recall:    {recall:.2f}')
  print(f'  - F1-Score:  {f1:.2f}')


# Run evaluation test
test_title = 'Naruto'
recs = recommend_anime(test_title, min_similarity=0.3, top_n=5)
evaluate_recommendation_quality(test_title, recs)

Training Set Size: 9613 rows
Testing Set Size: 2404 rows
Evaluation Metrics for recommendations of "Naruto":
  - Precision: 1.00
  - Recall:    1.00
  - F1-Score:  1.00


Evaluation Discussion & Analysis (Add in Markdown):

Genre Alignment Recommendations: The recommendations were able to successfully align genres (e.g., recommending action-adventure shounen anime for Naruto).

Areas for Improvement: Filtering based purely on genres lacks in understanding content-based filtering based on users’ behavioral tendencies. A combination of both TF-IDF genre-similarity and Collaborative Filtering (user rating matrix) would be much more effective.

**Task 5: Interview Questions & Core Concepts**

Essential concepts and interview answers regarding collaborative filtering and recommendation engines.


1. Please explain the difference between user-based and item-based collaborative filtering.
User-Based Collaborative Filtering: It works by looking for similar users who shared the same interests (e.g., “People who liked what you liked also liked…”). It narrows in on what the similar users have seen and liked. So, if you enjoy Anime A, you might like Anime B, too.
Item-Based Collaborative Filtering: It focuses on finding similar items (e.g., “Users who bought Anime A also bought Anime B…”). Two anime are assumed to be similar if they are regularly rated similarly by users who have watched both. It is often more efficient and scalable than user-based collaborative filtering, as item similarities tend to be more consistent over time.
---

2. What is collaborative filtering, and how does it work?
Definition: Collaborative filtering is a kind of recommendation system, which reads preference data from many users (allowing the system to generate an individualized recommendation for each user).
How it works: It works by assuming that if $A$ and $B$ have the same opinion about an issue, then $A$ will have the same opinion about a different issue as $B$ , rather than a randomly selected person with whom they have not interacted. Then, it uses a huge User-Item Interaction Matrix, which helps to fill empty cells by employing matrix factorization or nearest-neighbor algorithms.